In [12]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


# =============================================================================
# Загрузка базовых моделей (обучены на split1)
# =============================================================================

cat_model = joblib.load("model/cat_split1.joblib")
xgb_model = joblib.load("model/xgb_split1.joblib")
lgb_model = joblib.load("model/lgb_split1.joblib")

cat_model_full = joblib.load("model/cat_full.joblib")
xgb_model_full = joblib.load("model/xgb_full.joblib")
lgb_model_full = joblib.load("model/lgb_full.joblib")

# =============================================================================
# Загрузка данных
# Каждая модель может использовать свой набор признаков
# =============================================================================

CAT_FEATURES = "datasets/test_all_history_rolling_windows_velocity_trend_gmv.parquet"
XGB_FEATURES = "datasets/test_all_features.parquet"
LGB_FEATURES = "datasets/test_all_features.parquet"

all_ids = pd.read_parquet(CAT_FEATURES)[["user_id"]].copy()
train_ids, val_ids = train_test_split(all_ids, test_size=0.5, random_state=RANDOM_STATE)

def split_by_ids(df, ids):
  return df[df["user_id"].isin(ids["user_id"])]

VAL_CAT  = split_by_ids(pd.read_parquet(CAT_FEATURES), train_ids)
VAL2_CAT = split_by_ids(pd.read_parquet(CAT_FEATURES), val_ids)

VAL_XGB  = split_by_ids(pd.read_parquet(XGB_FEATURES), train_ids)
VAL2_XGB = split_by_ids(pd.read_parquet(XGB_FEATURES), val_ids)

VAL_LGB  = split_by_ids(pd.read_parquet(LGB_FEATURES), train_ids)
VAL2_LGB = split_by_ids(pd.read_parquet(LGB_FEATURES), val_ids)


def feat_cols(df):
    return [
        c for c in df.columns
        if c not in ("user_id", "target")
           and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def get_preds(model, df):
    return np.expm1(model.predict(df[feat_cols(df)]))


# =============================================================================
# Предсказания базовых моделей на split2 (train для мета-модели)
# =============================================================================

cat_preds_val = get_preds(cat_model, VAL_CAT)
xgb_preds_val = get_preds(xgb_model, VAL_XGB)
lgb_preds_val = get_preds(lgb_model, VAL_LGB)
y_val = VAL_CAT["target"].values

# =============================================================================
# Предсказания базовых моделей на split3 (val для мета-модели)
# =============================================================================

cat_preds_val2 = get_preds(cat_model, VAL2_CAT)
xgb_preds_val2 = get_preds(xgb_model, VAL2_XGB)
lgb_preds_val2 = get_preds(lgb_model, VAL2_LGB)
y_val2 = VAL2_CAT["target"].values


# =============================================================================
# Сборка мета-признаков
# =============================================================================

def meta_features(cat, xgb, lgb):
    return np.column_stack([
        np.log1p(cat),
        np.log1p(xgb),
        np.log1p(lgb),
    ])


X_meta_tr = meta_features(cat_preds_val, xgb_preds_val, lgb_preds_val)
X_meta_val = meta_features(cat_preds_val2, xgb_preds_val2, lgb_preds_val2)
y_meta_tr = np.log1p(y_val)
y_meta_val = np.log1p(y_val2)

# =============================================================================
# Мета-модель: Ridge
# =============================================================================

meta = Ridge(alpha=1.0, fit_intercept=True)
meta.fit(X_meta_tr, y_meta_tr)

meta_preds = np.expm1(meta.predict(X_meta_val))
meta_rmsle = rmsle(y_val2, meta_preds)

print(f"  CatBoost  val2 RMSLE : {rmsle(y_val2, cat_preds_val2):.5f}")
print(f"  XGBoost   val2 RMSLE : {rmsle(y_val2, xgb_preds_val2):.5f}")
print(f"  LightGBM  val2 RMSLE : {rmsle(y_val2, lgb_preds_val2):.5f}")
print(f"  Meta      val2 RMSLE : {meta_rmsle:.5f}")
print(f"\n  Meta weights — CB: {meta.coef_[0]:.4f}, XGB: {meta.coef_[1]:.4f}, LGB: {meta.coef_[2]:.4f}")

# =============================================================================
# Submission
# =============================================================================

SUB_CAT = pd.read_parquet("datasets/submission_all_history_rolling_windows_velocity_trend_gmv.parquet")
SUB_XGB = pd.read_parquet("datasets/submission_all_features.parquet")
SUB_LGB = pd.read_parquet("datasets/submission_all_features.parquet")

cat_sub = get_preds(cat_model_full, SUB_CAT)
xgb_sub = get_preds(xgb_model_full, SUB_XGB)
lgb_sub = get_preds(lgb_model_full, SUB_LGB)

X_meta_sub = meta_features(cat_sub, xgb_sub, lgb_sub)
ens_sub = np.expm1(meta.predict(X_meta_sub))

submission = pd.DataFrame({
    "user_id": SUB_CAT["user_id"].values,
    "predict": np.clip(ens_sub, 0, None),
})
submission.to_csv("submission_stacking.csv", index=False)

joblib.dump(meta, "model/meta_ridge.joblib")


['model/meta_ridge.joblib']

In [3]:
import joblib
import numpy as np
import pandas as pd
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


# =============================================================================
# Загрузка базовых моделей (обучены на split1)
# =============================================================================

cat_model = joblib.load("model/cat_split1.joblib")
xgb_model = joblib.load("model/xgb_split1.joblib")

# =============================================================================
# Загрузка данных
# =============================================================================

CAT_FEATURES = "datasets/split2_trend.parquet"
XGB_FEATURES = "datasets/split2_all_features.parquet"

VAL_CAT = pd.read_parquet(CAT_FEATURES)
VAL_XGB = pd.read_parquet(XGB_FEATURES)

VAL2_CAT = pd.read_parquet(CAT_FEATURES.replace("split2", "split3"))
VAL2_XGB = pd.read_parquet(XGB_FEATURES.replace("split2", "split3"))


def feat_cols(df):
    return [
        c for c in df.columns
        if c not in ("user_id", "target")
           and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def get_preds(model, df):
    return np.expm1(model.predict(df[feat_cols(df)]))


# =============================================================================
# Предсказания базовых моделей на split2 и split3
# =============================================================================

cat_preds_val = get_preds(cat_model, VAL_CAT)
xgb_preds_val = get_preds(xgb_model, VAL_XGB)
y_val = VAL_CAT["target"].values

cat_preds_val2 = get_preds(cat_model, VAL2_CAT)
xgb_preds_val2 = get_preds(xgb_model, VAL2_XGB)
y_val2 = VAL2_CAT["target"].values


# =============================================================================
# Подбор весов ансамбля через Optuna (на split2)
# =============================================================================

def objective(trial):
    w_cat = trial.suggest_float("w_cat", 0.0, 1.0)
    w_xgb = 1.0 - w_cat
    blend = w_cat * cat_preds_val + w_xgb * xgb_preds_val
    return rmsle(y_val, blend)


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=200, show_progress_bar=False)

best = study.best_params
w_cat = best["w_cat"]
w_xgb = 1.0 - w_cat

print(f"  Best weights — CatBoost: {w_cat:.4f}, XGBoost: {w_xgb:.4f}")
print(f"  Optuna train (split2) RMSLE : {study.best_value:.5f}")

# =============================================================================
# Оценка на split3
# =============================================================================

ens_val2 = w_cat * cat_preds_val2 + w_xgb * xgb_preds_val2

print(f"\n  CatBoost  val2 RMSLE : {rmsle(y_val2, cat_preds_val2):.5f}")
print(f"  XGBoost   val2 RMSLE : {rmsle(y_val2, xgb_preds_val2):.5f}")
print(f"  Ensemble  val2 RMSLE : {rmsle(y_val2, ens_val2):.5f}")

# =============================================================================
# Submission
# =============================================================================

SUB_CAT = pd.read_parquet("datasets/submission_trend.parquet")
SUB_XGB = pd.read_parquet("datasets/submission_all_features.parquet")

cat_sub = get_preds(cat_model, SUB_CAT)
xgb_sub = get_preds(xgb_model, SUB_XGB)

ens_sub = w_cat * cat_sub + w_xgb * xgb_sub

submission = pd.DataFrame({
    "user_id": SUB_CAT["user_id"].values,
    "predict": np.clip(ens_sub, 0, None),
})
submission.to_csv("submission_ensemble.csv", index=False)
print(f"\nSaved submission_ensemble.csv  ({len(submission)} rows)")

joblib.dump({"w_cat": w_cat, "w_xgb": w_xgb}, "model/ensemble_weights.joblib")
